# SteppeDNA: ESM-2 LoRA Fine-Tuning on HR Pathway Genes

This notebook fine-tunes the ESM-2 protein language model (650M) using LoRA (Low-Rank Adaptation) on
Homologous Recombination pathway gene sequences. The fine-tuned model produces better embeddings
for variant pathogenicity prediction.

**Requirements:** Google Colab with GPU (T4 is sufficient)

**What this does:**
1. Loads ESM-2 650M (esm2_t33_650M_UR50D)
2. Applies LoRA adapters to attention layers
3. Fine-tunes on HR pathway sequences using masked language modeling
4. Generates improved per-variant embeddings
5. Exports embeddings for use in SteppeDNA training pipeline

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers peft accelerate biopython pandas numpy scikit-learn

In [ ]:
# Step 2: Upload your master_training_dataset.csv
# Either drag-drop into Colab files panel, or use:
from google.colab import files
print("Upload master_training_dataset.csv from data/ folder:")
uploaded = files.upload()

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM, EsmModel
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 3: Define HR pathway gene sequences (reference)
# These are the canonical protein sequences for the 5 HR genes
GENE_SEQUENCES = {
    'BRCA1': None,  # Will be fetched from UniProt
    'BRCA2': None,
    'PALB2': None,
    'RAD51C': None,
    'RAD51D': None,
}

UNIPROT_IDS = {
    'BRCA1': 'P38398',
    'BRCA2': 'P51587',
    'PALB2': 'Q86YC2',
    'RAD51C': 'O43502',
    'RAD51D': 'O75771',
}

import requests

for gene, uid in UNIPROT_IDS.items():
    url = f'https://rest.uniprot.org/uniprotkb/{uid}.fasta'
    resp = requests.get(url)
    if resp.ok:
        lines = resp.text.strip().split('\n')
        seq = ''.join(lines[1:])  # skip header
        GENE_SEQUENCES[gene] = seq
        print(f'{gene}: {len(seq)} amino acids')
    else:
        print(f'FAILED to fetch {gene} ({uid})')

print(f'\nTotal sequences: {sum(1 for v in GENE_SEQUENCES.values() if v)}/5')

In [ ]:
# Step 4: Load ESM-2 650M with LoRA adapters
MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'

print('Loading ESM-2 650M tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Loading ESM-2 650M model...')
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# Apply LoRA to attention layers
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,  # Works for MLM too
    r=8,                           # LoRA rank
    lora_alpha=16,                 # Scaling factor
    lora_dropout=0.1,
    target_modules=['query', 'key', 'value'],  # Attention layers
    bias='none',
)

model = get_peft_model(model, lora_config)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\nTrainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Step 5: Create training dataset from HR pathway sequences
# We use sliding windows + random masking (MLM objective)

class HRSequenceDataset(Dataset):
    """Sliding window dataset over HR gene sequences for MLM fine-tuning."""
    def __init__(self, sequences, tokenizer, window_size=512, stride=256, mask_prob=0.15):
        self.tokenizer = tokenizer
        self.mask_prob = mask_prob
        self.windows = []
        
        for gene, seq in sequences.items():
            if seq is None:
                continue
            # Create overlapping windows
            for i in range(0, max(1, len(seq) - window_size + 1), stride):
                window = seq[i:i + window_size]
                self.windows.append((gene, window))
            # Also add the full sequence tail if not covered
            if len(seq) > window_size:
                self.windows.append((gene, seq[-window_size:]))
        
        print(f'Created {len(self.windows)} training windows from {len([s for s in sequences.values() if s])} genes')
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        gene, seq = self.windows[idx]
        encoding = self.tokenizer(seq, return_tensors='pt', truncation=True, 
                                   max_length=512, padding='max_length')
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        
        # Create MLM labels (mask random positions)
        labels = input_ids.clone()
        rand = torch.rand(input_ids.shape)
        mask_positions = (rand < self.mask_prob) & (attention_mask == 1)
        # Don't mask special tokens
        mask_positions[0] = False  # CLS
        eos_pos = (input_ids == self.tokenizer.eos_token_id).nonzero(as_tuple=True)
        if len(eos_pos[0]) > 0:
            mask_positions[eos_pos[0][0]] = False
        
        input_ids[mask_positions] = self.tokenizer.mask_token_id
        labels[~mask_positions] = -100  # Only compute loss on masked positions
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

dataset = HRSequenceDataset(GENE_SEQUENCES, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
# Step 6: Fine-tune with LoRA
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

EPOCHS = 10
LR = 2e-4

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler = GradScaler('cuda')

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    n_batches = 0
    for batch in dataloader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        
        with autocast('cuda'):
            outputs = model(**batch)
            loss = outputs.loss
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        n_batches += 1
    
    avg_loss = total_loss / max(n_batches, 1)
    print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}')

print('\nFine-tuning complete!')

In [ ]:
# Step 7: Generate per-variant embeddings using fine-tuned model
from sklearn.decomposition import PCA
import pickle

df = pd.read_csv('master_training_dataset.csv')
print(f'Loaded {len(df)} variants from master dataset')

# Detect the gene column name (could be 'gene' or 'gene_name')
gene_col = None
for col in ['gene_name', 'gene', 'Gene', 'Gene_Name']:
    if col in df.columns:
        gene_col = col
        break

if gene_col:
    print(f'Gene column: "{gene_col}"')
    print(f'Gene distribution:\n{df[gene_col].value_counts().to_string()}')
else:
    print('WARNING: No gene column found! Will treat all variants as BRCA2.')

GENE_AA_LENGTHS = {'BRCA1': 1863, 'BRCA2': 3418, 'PALB2': 1186, 'RAD51C': 376, 'RAD51D': 328}
CONTEXT_WINDOW = 50

# Switch to base model for embeddings (not MLM head)
model.eval()
base_model = model.base_model.model.esm  # Get the ESM backbone

all_embeddings = {}
batch_size = 32

for gene in ['BRCA1', 'BRCA2', 'PALB2', 'RAD51C', 'RAD51D']:
    if gene_col:
        gene_df = df[df[gene_col] == gene]
    else:
        gene_df = df if gene == 'BRCA2' else pd.DataFrame()
    
    if len(gene_df) == 0:
        print(f'{gene}: 0 variants, skipping')
        continue
    
    full_seq = GENE_SEQUENCES.get(gene)
    if not full_seq:
        print(f'{gene}: No reference sequence, skipping')
        continue
    
    gene_embeds = []
    gene_indices = []
    
    for idx, row in gene_df.iterrows():
        # Get AA position - try multiple column names
        aa_pos = None
        if 'relative_aa_pos' in row.index and pd.notna(row['relative_aa_pos']):
            aa_pos = int(row['relative_aa_pos'] * GENE_AA_LENGTHS[gene])
        elif 'AA_pos' in row.index and pd.notna(row['AA_pos']):
            aa_pos = int(row['AA_pos'])
        elif 'cDNA_pos' in row.index and pd.notna(row['cDNA_pos']):
            aa_pos = int(row['cDNA_pos']) // 3
        else:
            aa_pos = len(full_seq) // 2  # fallback to middle
        
        aa_pos = max(0, min(aa_pos, len(full_seq) - 1))
        
        # Extract context window
        start = max(0, aa_pos - CONTEXT_WINDOW)
        end = min(len(full_seq), aa_pos + CONTEXT_WINDOW + 1)
        context_seq = full_seq[start:end]
        
        gene_indices.append(idx)
        gene_embeds.append(context_seq)
    
    # Batch encode
    print(f'{gene}: Generating {len(gene_embeds)} embeddings...')
    embeddings_list = []
    
    for i in range(0, len(gene_embeds), batch_size):
        batch_seqs = gene_embeds[i:i+batch_size]
        inputs = tokenizer(batch_seqs, return_tensors='pt', padding=True, 
                          truncation=True, max_length=128)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = base_model(**inputs)
            hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings_list.append(pooled.cpu().numpy())
        
        if (i // batch_size) % 100 == 0 and i > 0:
            print(f'  {gene}: {i}/{len(gene_embeds)} done')
    
    gene_embedding_matrix = np.vstack(embeddings_list)
    
    for j, idx in enumerate(gene_indices):
        all_embeddings[idx] = gene_embedding_matrix[j]
    
    print(f'  -> {gene}: {gene_embedding_matrix.shape} done')

print(f'\nTotal embeddings: {len(all_embeddings)}')

In [ ]:
# Step 8: PCA reduction to 20 components (matching current pipeline)
indices = sorted(all_embeddings.keys())
embedding_matrix = np.array([all_embeddings[i] for i in indices])
print(f'Raw embedding shape: {embedding_matrix.shape}')

N_COMPONENTS = 20
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca_embeddings = pca.fit_transform(embedding_matrix)
print(f'PCA embeddings shape: {pca_embeddings.shape}')
print(f'Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Create a mapping: index -> PCA embedding
lora_embeddings = {}
for j, idx in enumerate(indices):
    lora_embeddings[idx] = pca_embeddings[j]

# Save
output = {
    'embeddings': lora_embeddings,
    'pca': pca,
    'model_name': MODEL_NAME,
    'lora_config': {'r': 8, 'alpha': 16, 'epochs': EPOCHS},
    'n_components': N_COMPONENTS,
    'variance_explained': float(pca.explained_variance_ratio_.sum()),
}

with open('esm2_lora_embeddings.pkl', 'wb') as f:
    pickle.dump(output, f)

print(f'\nSaved esm2_lora_embeddings.pkl')
print(f'Download this file and place it in SteppeDNA/data/')

In [ ]:
# Step 9: Save LoRA adapter weights (for reproducibility)
model.save_pretrained('esm2_lora_hr_adapter')

# Zip for download
!zip -r esm2_lora_hr_adapter.zip esm2_lora_hr_adapter/
print('Saved LoRA adapter weights to esm2_lora_hr_adapter.zip')

In [ ]:
# Step 10: Download outputs
from google.colab import files

files.download('esm2_lora_embeddings.pkl')
files.download('esm2_lora_hr_adapter.zip')

print('\n--- DONE ---')
print('Place esm2_lora_embeddings.pkl in SteppeDNA/data/')
print('Then retrain with: python scripts/train_universal_model.py')